# Reflex compilation

Goal: compile repeated validated decisions into a compact probabilistic reflex, then measure the tradeoff between coverage and correctness.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from sklearn.model_selection import train_test_split
from paradigm import ReflexCompiler, evaluate_reflex
from paradigm.synthetic import make_traces

traces = make_traces(5000, seed=1)
train, test = train_test_split(traces, test_size=0.25, random_state=1, stratify=[t.action for t in traces])
reflex = ReflexCompiler(random_state=1).fit(train, name="core-p0")
X_test = np.stack([t.features for t in test])
y_test = np.array([t.action for t in test])

In [ ]:
for threshold in [0.50, 0.70, 0.80, 0.90, 0.95, 0.98]:
    r = evaluate_reflex(reflex, X_test, y_test, accept_threshold=threshold)
    print(threshold, {
        "accuracy": round(r.accuracy, 4),
        "coverage": round(r.coverage, 4),
        "selective_accuracy": round(r.selective_accuracy, 4),
        "ece": round(r.ece, 4),
        "brier": round(r.brier, 4),
    })

## Required follow-up

Add exact cache, nearest-neighbor cache, decision tree, and uncalibrated random forest baselines before treating the compiled reflex as an improvement.